# Model Training and Evaluation

This notebook demonstrates:
- Training baseline and advanced models
- Model comparison
- Threshold optimization
- SHAP explainability

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from models.train_baseline import BaselineModel
from models.train_advanced import AdvancedModel
from models.threshold_tuning import ThresholdOptimizer
from models.explainability import ModelExplainer

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Train Baseline Model

In [ ]:
data_path = '../data/raw/accepted_2007_to_2018Q4.csv'

if Path(data_path).exists():
    baseline = BaselineModel()
    baseline_metrics = baseline.train(data_path)
    
    print("\nBaseline Model Metrics:")
    for dataset, metrics in baseline_metrics.items():
        print(f"\n{dataset}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
else:
    print(f"Data file not found at {data_path}")
    print("Please download the dataset first.")

## 2. Train Advanced Model

In [ ]:
if Path(data_path).exists():
    advanced = AdvancedModel()
    advanced_metrics = advanced.train(data_path, tune=False)
    
    print("\nAdvanced Model Metrics:")
    for dataset, metrics in advanced_metrics.items():
        print(f"\n{dataset}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")

## 3. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Baseline (Logistic)', 'Advanced (XGBoost)'],
    'Test F1': [
        baseline_metrics.get('Test', {}).get('f1', 0),
        advanced_metrics.get('Test', {}).get('f1', 0)
    ],
    'Test ROC-AUC': [
        baseline_metrics.get('Test', {}).get('roc_auc', 0),
        advanced_metrics.get('Test', {}).get('roc_auc', 0)
    ]
})

print("\nModel Comparison:")
print(comparison_df)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

comparison_df.plot(x='Model', y='Test F1', kind='bar', ax=ax[0], legend=False)
ax[0].set_title('F1 Score Comparison')
ax[0].set_ylabel('F1 Score')
ax[0].set_ylim([0, 1])

comparison_df.plot(x='Model', y='Test ROC-AUC', kind='bar', ax=ax[1], legend=False, color='orange')
ax[1].set_title('ROC-AUC Comparison')
ax[1].set_ylabel('ROC-AUC')
ax[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 4. Threshold Optimization

In [ ]:
model_path = '../models/final_model.joblib'
preprocessor_path = '../models/preprocessor.joblib'

if Path(model_path).exists() and Path(preprocessor_path).exists():
    import joblib
    
    model = joblib.load(model_path)
    preprocessor = joblib.load(preprocessor_path)
    
    sample_data = pd.read_csv(data_path, nrows=10000)
    
    sample_data = preprocessor.prepare_lending_club_data(sample_data)
    sample_data = preprocessor.engineer_features(sample_data)
    
    X_sample, y_sample = preprocessor.fit_transform(sample_data)
    
    y_proba = model.predict_proba(X_sample)[:, 1]
    
    optimizer = ThresholdOptimizer(cost_fp=1, cost_fn=5)
    optimal_threshold = optimizer.find_optimal_threshold(y_sample, y_proba, metric='cost')
    
    optimizer.plot_threshold_analysis()
else:
    print("Model not found. Please train the model first.")

## 5. SHAP Explainability

In [ ]:
if Path(model_path).exists() and Path(preprocessor_path).exists():
    explainer = ModelExplainer(model_path, preprocessor_path)
    
    sample_size = min(100, len(X_sample))
    X_explain = X_sample.head(sample_size)
    
    explainer.compute_shap_values(X_explain)
    
    print("\nGenerating SHAP summary plot...")
    explainer.plot_summary(X_explain, max_display=15)
    
    print("\nTop 10 Most Important Features:")
    top_features = explainer.get_top_features(X_explain, n_features=10)
    print(top_features)

## 6. Sample Prediction with Explanation

In [ ]:
if Path(model_path).exists():
    sample_loan = {
        'loan_amnt': 15000,
        'term': '36 months',
        'int_rate': 12.5,
        'installment': 500,
        'grade': 'C',
        'emp_length': '5 years',
        'annual_inc': 55000,
        'dti': 18.5,
        'delinq_2yrs': 0,
        'fico_range_high': 700,
        'revol_bal': 8000,
        'revol_util': 40.0
    }
    
    explanation, contributions = explainer.explain_prediction(sample_loan)
    
    print("\nSample Loan Prediction:")
    print("=" * 60)
    print(explanation)
    print("\nDetailed Feature Contributions:")
    print(contributions.head(10))

## Summary

This notebook demonstrated:
1. Training baseline and advanced models
2. Comparing model performance
3. Optimizing decision threshold for business costs
4. Understanding model predictions with SHAP

Next steps:
- Deploy models via API
- Monitor model performance in production
- Implement automated retraining